# Evaluation outputs on Colab (Tasks 2.9 + 2.10)

vLLM serves each model behind an OpenAI-compatible API; `eval/run.py` sends the 232 test prompts (same code as locally).
Runtime: **L4 or A100**. Use a fresh runtime (not the Unsloth training one).

**Cell 1 — install vLLM** (\"session crashed\" at the end is expected).

In [ ]:
!pip install -q vllm
!pip uninstall -y torchaudio
import os
os.kill(os.getpid(), 9)

**Cell 2 — Drive, repo, token, helpers** (run after every restart).

In [ ]:
import os, time, subprocess, httpx
from google.colab import drive, userdata
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

REPO = "/content/bureaucracy-navigator"
if not os.path.exists(REPO):
    !git clone https://github.com/AhmadTawil1/bureaucracy-navigator.git {REPO}
else:
    !git -C {REPO} pull
%cd {REPO}
DRIVE_RESULTS = "/content/drive/MyDrive/bureaucracy-navigator/eval_results"
os.makedirs(DRIVE_RESULTS, exist_ok=True)

server = None

def start_server(args: str, log: str):
    # vLLM in the background; waits until it answers /health (loading takes a few minutes).
    global server
    stop_server()
    server = subprocess.Popen(f"vllm serve {args} --port 8000 > {log} 2>&1", shell=True)
    for _ in range(180):
        try:
            if httpx.get("http://localhost:8000/health", timeout=2).status_code == 200:
                print("server ready"); return
        except httpx.HTTPError:
            pass
        if server.poll() is not None:
            break
        time.sleep(5)
    !tail -n 30 {log}
    raise RuntimeError("vLLM did not start, see the log above")

def stop_server():
    global server
    if server is not None:
        !pkill -f "vllm serve"
        server.wait()
        server = None
        time.sleep(5)

## Task 2.9 — Base and fine-tuned (one vLLM server: base model + your LoRA)

In [ ]:
from huggingface_hub import snapshot_download

LORA = snapshot_download("AhmadTawil1/dictalm3-bureaucracy-lora", token=os.environ["HF_TOKEN"])
start_server(f"dicta-il/DictaLM-3.0-1.7B-Instruct --enable-lora --lora-modules bureaucracy={LORA} "
             f"--max-lora-rank 16 --max-model-len 8192", "/content/vllm_student.log")

In [ ]:
!python -m eval.run generate --label base --base-url http://localhost:8000/v1 --model dicta-il/DictaLM-3.0-1.7B-Instruct
!python -m eval.run generate --label finetuned --base-url http://localhost:8000/v1 --model bureaucracy
!cp eval/results/base.jsonl eval/results/finetuned.jsonl {DRIVE_RESULTS}/

## Task 2.10 — Teacher (12B, with the tokenizer fix from `data/teacher.py`)

In [ ]:
from data.teacher import TEACHER_MODEL, fixed_tokenizer

start_server(f"{TEACHER_MODEL} --tokenizer {fixed_tokenizer()} --trust-remote-code "
             f"--max-model-len 8192 --gpu-memory-utilization 0.85", "/content/vllm_teacher.log")
!python -m eval.run generate --label teacher --base-url http://localhost:8000/v1 --model {TEACHER_MODEL}
!cp eval/results/teacher.jsonl {DRIVE_RESULTS}/
stop_server()